# Definição e cálculo dos KPIs — BPS (2020-2026)

**Sprint 3 — Definição das métricas e dos KPIs**

Este notebook define, calcula e valida os 6 KPIs obrigatórios a partir do dataset consolidado, antes de replicá-los como medidas DAX no Power BI.

In [1]:
import pandas as pd
from pathlib import Path

def encontrar_raiz_projeto(inicio: Path, marcador=("data", "raw")) -> Path:
    atual = inicio.resolve()
    for pasta in [atual] + list(atual.parents):
        if (pasta.joinpath(*marcador)).exists():
            return pasta
    raise FileNotFoundError(f"Não encontrei 'data/raw' subindo a partir de {inicio}")

RAIZ_PROJETO = encontrar_raiz_projeto(Path.cwd())
caminho_consolidado = RAIZ_PROJETO / "data" / "processed" / "BPS_20_26_Waldinei.csv"

df = pd.read_csv(caminho_consolidado, sep=";", encoding="utf-8")

print("Formato:", df.shape)
print(df.dtypes.head(10))

Formato: (367003, 37)
ano_compra            int64
cnpj_instituicao      int64
sg_uf                object
ds_esfera            object
dt_compra            object
dt_insercao          object
validade_compra       int64
co_catmat             int64
ds_item              object
co_pdm              float64
dtype: object


## Carregando o dataset consolidado

Ao salvar o dataset em CSV na Sprint 2, os tipos especiais que havíamos configurado no pandas (`datetime64` para as datas, `Int64` anulável para os códigos, texto com zero à esquerda para os CNPJs) não são preservados — CSV é um formato de texto puro, sem metadados de tipo. Isso significa que, ao reler o arquivo, o pandas reinterpreta cada coluna do zero.

Um problema concreto disso: **os CNPJs voltam a ser lidos como número**, perdendo de novo os zeros à esquerda corrigidos anteriormente (ex.: um CNPJ seria lido como `123456000199` em vez de `00123456000199`). Se não corrigido aqui, isso comprometeria a contagem de instituições/fornecedores distintos mais adiante.

Por isso, a leitura abaixo especifica explicitamente os tipos de cada coluna sensível (`dtype` para texto, `parse_dates` para datas), em vez de deixar o pandas inferir sozinho.

In [2]:
colunas_texto = ["cnpj_instituicao", "cnpj_fornecedor", "cnpj_fabricante"]
colunas_int_anulavel = ["co_pdm", "co_grupo", "co_classe", "registro_anvisa"]

df = pd.read_csv(
    caminho_consolidado,
    sep=";",
    encoding="utf-8",
    dtype={coluna: str for coluna in colunas_texto},
    parse_dates=["dt_compra", "dt_insercao"],
)

for coluna in colunas_int_anulavel:
    df[coluna] = df[coluna].astype("Int64")

print("Formato:", df.shape)
print(df.dtypes)

print("\n--- Verificação: CNPJs voltaram a ter 14 dígitos? ---")
for coluna in colunas_texto:
    tamanhos = df[coluna].str.len()
    print(f"{coluna}: tamanhos únicos -> {sorted(tamanhos.unique())}")

Formato: (367003, 37)
ano_compra                       int64
cnpj_instituicao                object
sg_uf                           object
ds_esfera                       object
dt_compra               datetime64[ns]
dt_insercao             datetime64[ns]
validade_compra                  int64
co_catmat                        int64
ds_item                         object
co_pdm                           Int64
co_grupo                         Int64
no_grupo                        object
co_classe                        Int64
no_classe                       object
fg_generico                     object
tp_compra                       object
sg_unidade_medida               object
cnpj_fornecedor                 object
no_fornecedor                   object
cnpj_fabricante                 object
no_fabricante                   object
qt_medicamento                   int64
ds_observacao                   object
no_instituicao                  object
no_municipio                    object
un_

Confirmado: os três CNPJs voltam a ter 14 dígitos após a leitura explícita de tipos. Dataset pronto para o cálculo dos KPIs.

## KPI 1, 2 e 3 — Valor total, quantidade total e número de registros

Os três primeiros KPIs são agregações diretas sobre o dataset consolidado, sem necessidade de tratamento especial.

In [3]:
valor_total_registrado = df["vl_preco_total"].sum()
quantidade_total_itens = df["qt_medicamento"].sum()
numero_registros = len(df)

print(f"1. Valor total registrado: R$ {valor_total_registrado:,.2f}")
print(f"2. Quantidade total de itens comprados: {quantidade_total_itens:,.0f}")
print(f"3. Número de registros de compra: {numero_registros:,.0f}")

1. Valor total registrado: R$ 115,063,593,346.89
2. Quantidade total de itens comprados: 64,807,253,018
3. Número de registros de compra: 367,003


In [4]:
print("Estatísticas de qt_medicamento:")
print(df["qt_medicamento"].describe())

print("\n--- Os 10 maiores valores de qt_medicamento ---")
print(df.nlargest(10, "qt_medicamento")[["ano_compra", "ds_item", "qt_medicamento", "vl_preco_unitario", "vl_preco_total", "no_instituicao"]])

Estatísticas de qt_medicamento:
count    3.670030e+05
mean     1.765851e+05
std      8.579067e+06
min      1.000000e+00
25%      2.500000e+02
50%      1.800000e+03
75%      1.200000e+04
max      4.356140e+09
Name: qt_medicamento, dtype: float64

--- Os 10 maiores valores de qt_medicamento ---
        ano_compra                                            ds_item  \
282690        2023  DIETA ENTERAL, ASPECTO FÍSICO:LÍQUIDO, USO:ENT...   
361560        2026            AMITRIPTILINA CLORIDRATO, DOSAGEM:25 MG   
202324        2022  DIETA ENTERAL, ASPECTO FÍSICO:LÍQUIDO, USO:ENT...   
274834        2023  DIETA ENTERAL, ASPECTO FÍSICO:LÍQUIDO, USO:ENT...   
211388        2022  DIETA ENTERAL, ASPECTO FÍSICO:LÍQUIDO, USO:ENT...   
233291        2022         CALCIPOTRIOL, DOSAGEM:50 MCG/G, USO:POMADA   
226085        2022                   HIDROCLOROTIAZIDA, DOSAGEM:25 MG   
29934         2020                        GABAPENTINA, DOSAGEM:400 MG   
159496        2021                        GABAPEN

In [5]:
colunas_verificar = ["ds_item", "qt_medicamento", "sg_unidade_medida", "un_fornecimento", "vl_preco_unitario", "vl_preco_total"]
print(df.nlargest(10, "qt_medicamento")[colunas_verificar].to_string())

                                                                                                                                                                                                                                                                                                                                                                                                                ds_item  qt_medicamento sg_unidade_medida un_fornecimento  vl_preco_unitario  vl_preco_total
282690                             DIETA ENTERAL, ASPECTO FÍSICO:LÍQUIDO, USO:ENTERAL OU ORAL, CARACTERÍSTICAS:HIPERCALÓRICA,NORMOPROTEICA, FONTE DE PROTEÍNA:CASEINATO E/OU PTN SOJA E/OU SORO LEITE, FONTE DE CARBOIDRATO:MALTODEXTRINA, FONTE DE LIPÍDIOS:ÓLEOS VEG.E/OU TCM E/OU LEC.SOJA, COMPONENTES ADICIONAIS:AA'S,VIT.,MINERAIS, CARACTERÍSTICAS ADICIONAIS:ISENTO GLÚTEN,LACT.,SACAROSE, SABOR:C/ OU S/ SABOR      4356140000               NaN       MILILITRO             0.0125      54451750.0
361560        

In [6]:
# Quanto os 10 maiores valores de qt_medicamento representam do total?
top10 = df.nlargest(10, "qt_medicamento")
print("Soma dos 10 maiores:", top10["qt_medicamento"].sum())
print("Total geral:", df["qt_medicamento"].sum())
print(f"Percentual do total: {top10['qt_medicamento'].sum() / df['qt_medicamento'].sum() * 100:.2f}%")

# O mesmo, mas separando só os que NÃO são líquidos medidos em mL (os realmente suspeitos)
suspeitos = df[
    (df["qt_medicamento"] > 100_000_000) &
    (~df["un_fornecimento"].isin(["MILILITRO", "LITRO"]))
]
print(f"\nRegistros suspeitos (>100 milhões, não líquidos): {len(suspeitos)}")
print("Soma desses registros:", suspeitos["qt_medicamento"].sum())
print(f"Percentual do total geral: {suspeitos['qt_medicamento'].sum() / df['qt_medicamento'].sum() * 100:.2f}%")

Soma dos 10 maiores: 10885884000
Total geral: 64807253018
Percentual do total: 16.80%

Registros suspeitos (>100 milhões, não líquidos): 31
Soma desses registros: 8000359366
Percentual do total geral: 12.34%


In [7]:
# Cálculo COM todos os dados
preco_medio_ponderado_completo = df["vl_preco_total"].sum() / df["qt_medicamento"].sum()

# Cálculo excluindo os 31 registros suspeitos
df_sem_suspeitos = df[~df.index.isin(suspeitos.index)]
preco_medio_ponderado_sem_suspeitos = (
    df_sem_suspeitos["vl_preco_total"].sum() / df_sem_suspeitos["qt_medicamento"].sum()
)

print(f"Preço médio ponderado (todos os dados): R$ {preco_medio_ponderado_completo:.4f}")
print(f"Preço médio ponderado (sem os 31 suspeitos): R$ {preco_medio_ponderado_sem_suspeitos:.4f}")
print(f"Diferença: {abs(preco_medio_ponderado_completo - preco_medio_ponderado_sem_suspeitos) / preco_medio_ponderado_sem_suspeitos * 100:.2f}%")

# Quais itens específicos são esses 31?
print("\n--- Itens dos registros suspeitos ---")
print(suspeitos[["ano_compra", "ds_item", "qt_medicamento", "un_fornecimento", "vl_preco_unitario"]].to_string())

Preço médio ponderado (todos os dados): R$ 1.7755
Preço médio ponderado (sem os 31 suspeitos): R$ 1.9958
Diferença: 11.04%

--- Itens dos registros suspeitos ---
        ano_compra                                                                                                                                                                                                                                                                                                                                                                                                 ds_item  qt_medicamento un_fornecimento  vl_preco_unitario
29934         2020                                                                                                                                                                                                                                                                                                                                                                      

## Observação: concentração de volume em poucos registros

Um pequeno grupo de registros (31 de 367.003, ou 0,008%) concentra 12,3% da quantidade total comprada — em sua maioria medicamentos de altíssimo consumo no SUS (Losartana, Hidroclorotiazida, Dipirona, Salbutamol), consistentes com compras centralizadas de grande escala, e não com erro aparente de digitação (os valores variam de forma plausível entre anos, sem repetição suspeita — exceto Gabapentina 400mg, idêntica em 2020 e 2021).

**Decisão:** os dados foram mantidos sem exclusão, por não haver evidência suficiente de erro. Registra-se como limitação que o KPI de preço unitário médio ponderado é sensível a essa concentração: o valor calculado com o dataset completo (R$ 1,78) difere em 11% do valor calculado excluindo esses registros (R$ 2,00).

## KPI 4, 5 e 6 — Instituições, fornecedores e preço unitário médio ponderado

In [8]:
instituicoes_compradoras = df["cnpj_instituicao"].nunique()
fornecedores = df["cnpj_fornecedor"].nunique()
preco_unitario_medio_ponderado = df["vl_preco_total"].sum() / df["qt_medicamento"].sum()

print(f"4. Instituições compradoras (CNPJs distintos): {instituicoes_compradoras}")
print(f"5. Fornecedores (CNPJs distintos): {fornecedores}")
print(f"6. Preço unitário médio ponderado: R$ {preco_unitario_medio_ponderado:.4f}")

4. Instituições compradoras (CNPJs distintos): 854
5. Fornecedores (CNPJs distintos): 3663
6. Preço unitário médio ponderado: R$ 1.7755


## Resumo dos 6 KPIs obrigatórios

| KPI | Fórmula | Valor (2020-2026) |
|---|---|---|
| Valor total registrado | `SUM(vl_preco_total)` | R$ 115.063.593.346,89 |
| Quantidade total de itens comprados | `SUM(qt_medicamento)` | 64.807.253.018 |
| Número de registros de compra | `COUNT(*)` | 367.003 |
| Instituições compradoras | `COUNT(DISTINCT cnpj_instituicao)` | 854 |
| Fornecedores | `COUNT(DISTINCT cnpj_fornecedor)` | 3.663 |
| Preço unitário médio ponderado | `SUM(vl_preco_total) / SUM(qt_medicamento)` | R$ 1,7755 |

**Nota sobre o preço unitário médio ponderado:** calculado como a razão entre os dois totais (não como média aritmética de `vl_preco_unitario`), conforme instrução do desafio — evita distorção por diferenças de escala entre compras pequenas e grandes. Sensível à concentração de volume discutida acima (variação de 11% se os 31 registros de alto volume forem excluídos).